In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

def aplicar_filtro_homomorfico(img_path, gL, gH, D0, c):
    # Cargar la imagen
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        print(f"⚠️ Error: No se encontró la imagen '{img_path}'.")
        return

    # ---------------------------------------------------------
    # PIPELINE HOMOMÓRFICO: f -> ln -> F -> H -> F^-1 -> exp -> g
    # ---------------------------------------------------------
    # 1. Logaritmo natural (np.log1p equivale a log(1 + x) para evitar log(0))
    # Pasamos a float32 para no saturar valores
    img_log = np.log1p(np.float32(img))
    
    # 2. TDF y centrado
    fshift = np.fft.fftshift(np.fft.fft2(img_log))
    
    # 3. Generación del filtro H(u,v) (Gaussiano modificado)
    filas, columnas = img.shape
    x, y = np.ogrid[:filas, :columnas]
    centro_x, centro_y = filas // 2, columnas // 2
    
    # Distancia al cuadrado
    D2 = (x - centro_x)**2 + (y - centro_y)**2
    
    # Para evitar división por cero en el centro
    D0 = max(D0, 1e-5) 
    
    # Filtro Homomórfico: (gH - gL) * [1 - exp(-c * (D^2 / D0^2))] + gL
    H = (gH - gL) * (1 - np.exp(-c * (D2 / (D0**2)))) + gL
    
    # 4. Filtrado (Producto en frecuencia)
    fshift_filtrado = fshift * H
    
    # 5. TDF Inversa
    img_filtrada_log = np.real(np.fft.ifft2(np.fft.ifftshift(fshift_filtrado)))
    
    # 6. Exponencial (expm1 equivale a exp(x) - 1, revierte el log1p)
    img_filtrada = np.expm1(img_filtrada_log)
    
    # 7. Normalización a 0-255 uint8 para poder visualizar
    img_homomorfico = cv2.normalize(img_filtrada, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)

    # ---------------------------------------------------------
    # COMPARATIVAS
    # ---------------------------------------------------------
    # Ecualización Clásica (Sobre la original)
    img_ecualizada = cv2.equalizeHist(img)
    
    # Homomórfico + Ecualización
    img_combo = cv2.equalizeHist(img_homomorfico)

    # ---------------------------------------------------------
    # VISUALIZACIÓN
    # ---------------------------------------------------------
    fig, axs = plt.subplots(2, 2, figsize=(16, 12))
    
    axs[0, 0].imshow(img, cmap='gray')
    axs[0, 0].set_title('1. Imagen Original (Iluminación pobre)')
    axs[0, 0].axis('off')
    
    axs[0, 1].imshow(img_ecualizada, cmap='gray')
    axs[0, 1].set_title('2. Ecualización Clásica\n(Suele "quemar" las zonas claras)')
    axs[0, 1].axis('off')
    
    axs[1, 0].imshow(img_homomorfico, cmap='gray')
    axs[1, 0].set_title(f'3. Filtro Homomórfico\n(Aclara sombras protegiendo brillos)')
    axs[1, 0].axis('off')
    
    axs[1, 1].imshow(img_combo, cmap='gray')
    axs[1, 1].set_title('4. Homomórfico + Ecualizada\n(El mejor resultado global)')
    axs[1, 1].axis('off')
    
    plt.tight_layout()
    plt.show()

print("--- EJERCICIO 4: Sintonizador de Filtrado Homomórfico ---")

interact(aplicar_filtro_homomorfico,
         img_path=Dropdown(options=['casilla.tif', 'reunion.tif'], value='casilla.tif', description='Imagen:'),
         # Agregamos la "r" antes de las comillas en las descripciones
         gL=FloatSlider(min=0.0, max=1.0, step=0.1, value=0.3, description=r'Gamma L ', continuous_update=False),
         gH=FloatSlider(min=1.0, max=5.0, step=0.1, value=1.5, description=r'Gamma H ', continuous_update=False),
         D0=IntSlider(min=1, max=100, step=1, value=30, description=r'Corte ', continuous_update=False),
         c=FloatSlider(min=0.1, max=5.0, step=0.1, value=1.0, description=r'Constante ', continuous_update=False)
)

--- EJERCICIO 4: Sintonizador de Filtrado Homomórfico ---


interactive(children=(Dropdown(description='Imagen:', options=('casilla.tif', 'reunion.tif'), value='casilla.t…

<function __main__.aplicar_filtro_homomorfico(img_path, gL, gH, D0, c)>